# AutoGen Studio on Colab — local models + Cloudflare tunnel


In [1]:
#@title 1. Setup (paths + helpers)
#@markdown `latest` = newest stable AutoGen Studio on PyPI. Or pin one (e.g. `0.4.2.2`), or use `pre` for the newest pre-release.
AGS_VERSION = "latest" #@param {type:"string"}
import os, json, secrets, subprocess, socket, time, re, shutil

BASE   = "/content"
DATA   = f"{BASE}/autogen-data"   # everything worth backing up
APPDIR = f"{DATA}/appdir"         # AutoGen Studio database, files, team configs
LOGS   = f"{DATA}/logs"
VENV   = f"{BASE}/autogen-env"    # AutoGen Studio gets its own Python env
STUDIO_PORT = 8081
for d in (LOGS, f"{APPDIR}/configs"):
    os.makedirs(d, exist_ok=True)

_kf = f"{DATA}/keys.json"
KEYS = json.load(open(_kf)) if os.path.exists(_kf) else {}
KEYS.setdefault("ACCESS_KEY", secrets.token_hex(16))
with open(_kf, "w") as fh:
    json.dump(KEYS, fh)

os.environ.update(AGS_VERSION_REQ=AGS_VERSION.strip() or "latest", AGS_VENV=VENV)

def bg(name, cmd, env=None, cwd=None):
    """Start a long-running process in the background, logging to LOGS/name.log."""
    log = open(f"{LOGS}/{name}.log", "a")
    return subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                            stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

def wait_port(port, name, timeout=300):
    t = time.time()
    while time.time() - t < timeout:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                print(f"{name} is up on :{port}"); return True
        time.sleep(2)
    print(f"{name} did not start on :{port} — check {LOGS}/{name}.log"); return False

print("Settings saved.")

Settings saved.


In [2]:
%%bash
#@title 2. Install cloudflared, nginx and AutoGen Studio (safe to re-run)
set -e
export DEBIAN_FRONTEND=noninteractive
AGS_VENV="${AGS_VENV:-/content/autogen-env}"

fetch() { curl -fsSL -o "$2.tmp" "$1" && mv -f "$2.tmp" "$2"; }

if ! command -v nginx >/dev/null; then
  apt-get -qq update 2>/dev/null || true
  apt-get -qq install -y nginx ca-certificates curl > /dev/null
fi

if ! command -v cloudflared >/dev/null; then
  fetch https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 /usr/local/bin/cloudflared
  chmod +x /usr/local/bin/cloudflared
fi

command -v uv >/dev/null || pip -q install -U uv

V="${AGS_VERSION_REQ:-latest}"
case "$V" in
  latest) SPEC="autogenstudio" ;;
  pre)    SPEC="autogenstudio>=0.4.0.dev0" ;;
  *)      SPEC="autogenstudio==${V#v}" ;;
esac

[ -x "$AGS_VENV/bin/python" ] || uv venv -q --python 3.12 "$AGS_VENV"
echo "▶ Installing $SPEC (1–3 min the first time)..."
uv pip install -q --python "$AGS_VENV/bin/python" -U "$SPEC" "autogen-ext[openai]"

"$AGS_VENV/bin/python" - <<'EOF'
import importlib.metadata as m
print(" | ".join(f"{p} {m.version(p)}" for p in ("autogenstudio", "autogen-agentchat", "autogen-ext", "autogen-core")))
EOF
echo "$(cloudflared --version 2>&1 | head -1)"

▶ Installing autogenstudio (1–3 min the first time)...
autogenstudio 0.4.2.2 | autogen-agentchat 0.5.7 | autogen-ext 0.5.7 | autogen-core 0.5.7
cloudflared version 2026.10.0 (built 2026-10-05-17:37 UTC)


In [3]:
#@title 3. Local model (Ollama or vLLM) — needs a GPU runtime
#@markdown Only one engine runs at a time (a T4 can't hold both). Re-run this cell to switch engine or model, then re-run cell 4.
ENGINE = "ollama" #@param ["ollama", "vllm"]
#@markdown **Ollama**: any tag from ollama.com/library. Pick one that supports tool calling — agents depend on it.
OLLAMA_MODEL = "qwen2.5:7b" #@param {type:"string"}
OLLAMA_NUM_CTX = 8192 #@param {type:"integer"}
#@markdown **vLLM**: a Hugging Face model id that fits the GPU (a free T4 has 15 GB). The tool parser must match the model family (`hermes` for Qwen); leave it empty to turn tool calling off.
VLLM_MODEL = "Qwen/Qwen2.5-3B-Instruct" #@param {type:"string"}
VLLM_MAX_LEN = 8192 #@param {type:"integer"}
VLLM_TOOL_PARSER = "hermes" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

import urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none — it will run on CPU (very slow; Ollama only)")

def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout

def _chat(m):
    """One small request through the OpenAI-compatible API, the same route AutoGen uses. Also loads the model."""
    req = urllib.request.Request(m["base_url"] + "/chat/completions", method="POST",
        data=json.dumps({"model": m["model"], "max_tokens": 16,
                         "messages": [{"role": "user", "content": "Reply with one word: ready"}]}).encode(),
        headers={"Content-Type": "application/json", "Authorization": "Bearer " + m["api_key"]})
    return json.load(urllib.request.urlopen(req, timeout=600))["choices"][0]["message"]["content"]

if ENGINE == "ollama":
    subprocess.run(["pkill", "-9", "-f", "vllm serve"])
    if not shutil.which("ollama"):
        print("▶ Installing Ollama...")
        _sh("apt-get -qq install -y zstd pciutils lshw > /dev/null && curl -fsSL https://ollama.com/install.sh | sh")
    subprocess.run(["pkill", "-9", "-f", "ollama (serve|runner)"])
    time.sleep(1)
    bg("ollama", "ollama serve", {"OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "-1",
                                  "OLLAMA_CONTEXT_LENGTH": str(OLLAMA_NUM_CTX),
                                  "OLLAMA_MODELS": f"{BASE}/ollama-models"})
    if not wait_port(11434, "ollama", 60):
        raise RuntimeError(f"Ollama didn't start — check {LOGS}/ollama.log")
    name = OLLAMA_MODEL.strip()
    print(f"▶ Pulling {name} (first time can take a few minutes)...")
    _sh(f"ollama pull {name}")
    MODEL = {"engine": "ollama", "model": name, "base_url": "http://127.0.0.1:11434/v1", "api_key": "ollama"}
else:
    subprocess.run(["pkill", "-9", "-f", "ollama (serve|runner)"])
    VLLM_ENV = f"{BASE}/vllm-env"
    if not os.path.exists(f"{VLLM_ENV}/bin/vllm"):
        print("▶ Installing vLLM (~5–10 min, one time)...")
        _sh(f"pip -q install -U uv && uv venv -q --python 3.12 {VLLM_ENV} && uv pip install -q --python {VLLM_ENV}/bin/python vllm")
    subprocess.run(["pkill", "-9", "-f", "vllm serve"])
    time.sleep(2)
    SERVED = VLLM_MODEL.strip().split("/")[-1]
    major = int(subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True,
                               capture_output=True, text=True).stdout.strip().split(".")[0] or 0)
    dtype = "half" if major < 8 else "auto"   # T4 (7.5) has no bfloat16
    env = {"HF_HOME": f"{BASE}/hf-cache"}
    if HF_TOKEN.strip(): env["HF_TOKEN"] = HF_TOKEN.strip()
    parser = VLLM_TOOL_PARSER.strip()
    tools = f"--enable-auto-tool-choice --tool-call-parser {parser}" if parser else ""
    bg("vllm", f"{VLLM_ENV}/bin/vllm serve {VLLM_MODEL.strip()} --host 127.0.0.1 --port 8000 --served-model-name {SERVED} "
               f"--dtype {dtype} --max-model-len {VLLM_MAX_LEN} --gpu-memory-utilization 0.85 {tools}", env)
    print("▶ Downloading + loading model (several minutes)...")
    for _ in range(450):   # up to 15 min
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5); break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError(f"vLLM didn't come up — check {LOGS}/vllm.log")
    MODEL = {"engine": "vllm", "model": SERVED, "base_url": "http://127.0.0.1:8000/v1", "api_key": "EMPTY"}

print("▶ Test request (loads the model into memory)...")
print("  ✅ model answered:", repr(_chat(MODEL)))
with open(f"{DATA}/model.json", "w") as fh:
    json.dump(MODEL, fh)
print(f"\n{MODEL['engine']} is serving '{MODEL['model']}' at {MODEL['base_url']}")
print("Next: run cell 4. If AutoGen Studio is already running, re-run cell 4 so it picks up this model.")

GPU: Tesla T4, 15360 MiB
▶ Installing Ollama...
ollama is up on :11434
▶ Pulling qwen2.5:7b (first time can take a few minutes)...
▶ Test request (loads the model into memory)...
  ✅ model answered: 'Ready'

ollama is serving 'qwen2.5:7b' at http://127.0.0.1:11434/v1
Next: run cell 4. If AutoGen Studio is already running, re-run cell 4 so it picks up this model.


In [4]:
#@title 4. Start AutoGen Studio + Cloudflare tunnel
#@markdown **Access** — AutoGen Studio has no login of its own and the tunnel URL is public. With this on, only people who have the secret link printed below get in.
REQUIRE_ACCESS_KEY = True #@param {type:"boolean"}
#@markdown **Keys (optional)** — only for Studio's built-in sample teams, which default to OpenAI models. The local teams don't need one.
OPENAI_API_KEY = "" #@param {type:"string"}
#@markdown **Stable URL (optional)** — a named Cloudflare tunnel token + its hostname. Point the tunnel at `http://127.0.0.1:8090`.
CF_TUNNEL_TOKEN = "" #@param {type:"string"}
CUSTOM_URL = "" #@param {type:"string"}
#@markdown Tick once after restoring a backup made with an older AutoGen Studio version.
UPGRADE_DATABASE = False #@param {type:"boolean"}

# Extra environment variables for AutoGen Studio, e.g. keys for other providers:
EXTRA_ENV = {
    # "ANTHROPIC_API_KEY": "sk-ant-...",
    # "AZURE_OPENAI_API_KEY": "...",
}

import psutil, urllib.request, urllib.error
from IPython.display import display, HTML

def _free(p):
    with socket.socket() as s: return s.connect_ex(("127.0.0.1", p)) != 0

def _kill_port(port):
    for c in psutil.net_connections(kind="tcp"):
        if c.laddr and c.laddr.port == port and c.status == psutil.CONN_LISTEN and c.pid:
            try:
                p = psutil.Process(c.pid)
                for ch in p.children(recursive=True): ch.kill()
                p.kill()
            except psutil.NoSuchProcess:
                pass

for pat in ["cloudflared tunnel", "autogenstudio ui"]:
    subprocess.run(["pkill", "-9", "-f", pat])
subprocess.run(["nginx", "-s", "stop"], stderr=subprocess.DEVNULL)
time.sleep(1)
for _ in range(15):
    if _free(STUDIO_PORT): break
    _kill_port(STUDIO_PORT); time.sleep(1)
else:
    raise RuntimeError(f"Port {STUDIO_PORT} is still busy")

# --- Teams that use the local model. Studio imports every JSON in <appdir>/configs when it starts,
# --- so they are built here with the same AutoGen version Studio runs on.
SEED = r'''
import json, sys
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

m, out = json.load(open(sys.argv[1])), sys.argv[2]
client = OpenAIChatCompletionClient(
    model=m["model"], base_url=m["base_url"], api_key=m["api_key"],
    model_info={"vision": False, "function_calling": True, "json_output": True,
                "structured_output": True, "family": "unknown"})

def save(team, slug, label, description):
    comp = team.dump_component()
    comp.label, comp.description = label, description
    with open(f"{out}/local_{slug}.json", "w") as f:
        json.dump(comp.model_dump(), f, indent=2,
                  default=lambda o: o.get_secret_value() if hasattr(o, "get_secret_value") else str(o))

tag = f"{m['engine']}: {m['model']}"
assistant = AssistantAgent("assistant", model_client=client,
    system_message="You are a helpful assistant. Work through the task step by step. "
                   "When the task is complete, end your reply with TERMINATE.")
save(RoundRobinGroupChat([assistant],
        termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(6)),
     "assistant", f"Local assistant ({tag})", "One assistant agent on the local model.")

writer = AssistantAgent("writer", model_client=client,
    system_message="You write clear, concise text for the given task and revise it when you get feedback.")
reviewer = AssistantAgent("reviewer", model_client=client,
    system_message="You review the writer's latest draft and give short, concrete feedback. "
                   "When the draft is good enough, reply with exactly: APPROVE")
save(RoundRobinGroupChat([writer, reviewer],
        termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8)),
     "writer_reviewer", f"Local writer + reviewer ({tag})", "Two agents taking turns on the local model.")
'''
CONFIGS = f"{APPDIR}/configs"
os.makedirs(CONFIGS, exist_ok=True)
for f in os.listdir(CONFIGS):
    if f.startswith("local_"): os.remove(f"{CONFIGS}/{f}")
_mf = f"{DATA}/model.json"
MODEL = json.load(open(_mf)) if os.path.exists(_mf) else None
if MODEL:
    with open(f"{DATA}/seed_teams.py", "w") as fh:
        fh.write(SEED)
    r = subprocess.run([f"{VENV}/bin/python", f"{DATA}/seed_teams.py", _mf, CONFIGS], capture_output=True, text=True)
    print("local-model teams written" if r.returncode == 0
          else "⚠️ Couldn't write the local-model teams:\n" + r.stderr[-1500:])
else:
    print("No local model yet (cell 3) — only Studio's built-in sample teams will be listed.")

env = {k: str(v) for k, v in EXTRA_ENV.items() if str(v).strip()}
if OPENAI_API_KEY.strip(): env["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
print("▶ Starting AutoGen Studio...")
bg("studio", f"{VENV}/bin/autogenstudio ui --host 127.0.0.1 --port {STUDIO_PORT} --appdir {APPDIR}"
             + (" --upgrade-database" if UPGRADE_DATABASE else ""), env)

# --- nginx: websocket-aware proxy + the secret-link gate
KEY = KEYS["ACCESS_KEY"]
NGINX_PORT = next(p for p in range(8090, 8200) if _free(p))
NGINX = """map $http_upgrade $ags_conn { default upgrade; '' close; }
map $cookie_ags_access $ags_ok { default __OPEN__; "__KEY__" 1; }
server {
    listen 127.0.0.1:__PORT__;
    client_max_body_size 100M;
    default_type text/plain;
    absolute_redirect off;

    location = /_open {
        if ($arg_key != "__KEY__") { return 403 "Wrong or missing access key."; }
        add_header Set-Cookie "ags_access=__KEY__; Path=/; Max-Age=2592000; HttpOnly; Secure; SameSite=Lax" always;
        return 302 /;
    }
    location / {
        if ($ags_ok = 0) { return 403 "Open the access link printed by notebook cell 4 first."; }
        proxy_pass http://127.0.0.1:__STUDIO__;
        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection $ags_conn;
        proxy_set_header Host $host;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto https;
        proxy_buffering off;
        proxy_cache off;
        proxy_read_timeout 3600s;
        proxy_send_timeout 3600s;
    }
}
"""
for k, v in {"__PORT__": NGINX_PORT, "__STUDIO__": STUDIO_PORT, "__KEY__": KEY,
             "__OPEN__": 0 if REQUIRE_ACCESS_KEY else 1}.items():
    NGINX = NGINX.replace(k, str(v))
with open("/etc/nginx/sites-available/autogen", "w") as fh:
    fh.write(NGINX)
subprocess.run("rm -f /etc/nginx/sites-enabled/default && ln -sf /etc/nginx/sites-available/autogen /etc/nginx/sites-enabled/autogen", shell=True)
r = subprocess.run("nginx -t && nginx", shell=True, capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("nginx failed:\n" + r.stderr)

ok = all([wait_port(STUDIO_PORT, "studio", 240), wait_port(NGINX_PORT, "nginx", 30)])

# --- Cloudflare tunnel
open(f"{LOGS}/cloudflared.log", "w").close()
if CF_TUNNEL_TOKEN.strip():
    assert CUSTOM_URL.strip().startswith("https://"), "Set CUSTOM_URL to the tunnel's https:// hostname"
    bg("cloudflared", f"cloudflared tunnel --no-autoupdate run --token {CF_TUNNEL_TOKEN.strip()}")
    PUBLIC_URL = CUSTOM_URL.strip().rstrip("/")
else:
    bg("cloudflared", f"cloudflared tunnel --no-autoupdate --url http://127.0.0.1:{NGINX_PORT}")
    PUBLIC_URL = None
    for _ in range(60):
        found = [u for u in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{LOGS}/cloudflared.log").read())
                 if not u.startswith("https://api.")]
        if found:
            PUBLIC_URL = found[0]; break
        time.sleep(1)
    assert PUBLIC_URL, f"No tunnel URL yet — see {LOGS}/cloudflared.log"
OPEN_URL = f"{PUBLIC_URL}/_open?key={KEY}" if REQUIRE_ACCESS_KEY else PUBLIC_URL
print("Public URL:", PUBLIC_URL, "| nginx port:", NGINX_PORT)

if ok:
    ver = subprocess.run([f"{VENV}/bin/python", "-c", "import importlib.metadata as m; print(m.version('autogenstudio'))"],
                         capture_output=True, text=True).stdout.strip()
    local = (f"<p>Local model: <code>{MODEL['model']}</code> via {MODEL['engine']}. In Studio open <b>Playground</b>, "
             f"start a new session and pick a team whose name starts with <b>Local</b>.</p>"
             if MODEL else "<p>No local model configured — run cell 3, then this cell again.</p>")
    gate = ("<p>This link carries your access key, so share it only with people you trust. "
            "It sets a cookie; after the first visit the plain URL works in that browser too.</p>"
            if REQUIRE_ACCESS_KEY else
            "<p>⚠️ The access key is off: anyone with this URL can use the Studio and run agents on this runtime.</p>")
    display(HTML(f"""<h2>🎉 AutoGen Studio {ver} is running</h2>
    <p>Open: <a href="{OPEN_URL}" target="_blank">{OPEN_URL}</a></p>{gate}{local}"""))
else:
    print(f"Something didn't start — run the Status cell and read {LOGS}/studio.log")

local-model teams written
▶ Starting AutoGen Studio...
studio is up on :8081
nginx is up on :8090
Public URL: https://tracy-resolutions-derby-workplace.trycloudflare.com | nginx port: 8090


In [5]:
#@title 5. Status & logs
SERVICE = "studio" #@param ["studio", "cloudflared", "ollama", "vllm"]
LINES = 40 #@param {type:"integer"}
import urllib.request, urllib.error
def _up(port):
    with socket.socket() as s: return s.connect_ex(("127.0.0.1", port)) == 0
for port, name in [(STUDIO_PORT, "studio"), (NGINX_PORT, "nginx"), (11434, "ollama"), (8000, "vllm")]:
    print(f"{'🟢' if _up(port) else '🔴'} {name:14} :{port}")
try:
    urllib.request.urlopen(PUBLIC_URL, timeout=15)
    print(f"🟢 {'tunnel':14} {PUBLIC_URL}")
except urllib.error.HTTPError:
    print(f"🟢 {'tunnel':14} {PUBLIC_URL}")
except Exception as e:
    print(f"🔴 {'tunnel':14} {PUBLIC_URL}  ({e})")
print(f"\nOpen: {OPEN_URL}")
print(f"\n--- last {LINES} lines of {SERVICE}.log ---")
print(subprocess.run(["tail", "-n", str(LINES), f"{LOGS}/{SERVICE}.log"], capture_output=True, text=True).stdout)

🟢 studio         :8081
🟢 nginx          :8090
🟢 ollama         :11434
🔴 vllm           :8000
🔴 tunnel         https://tracy-resolutions-derby-workplace.trycloudflare.com  (<urlopen error [Errno -2] Name or service not known>)

Open: https://tracy-resolutions-derby-workplace.trycloudflare.com/_open?key=dd5db40e60d0ce7452b76551bcf3dba3

--- last 40 lines of studio.log ---
INFO:     Loading environment from '/root/.autogenstudio/temp_env_vars.env'
2026-10-11 11:14:28.634 | INFO     | autogenstudio.web.initialization:__init__:39 - Initializing application data folder: /content/autogen-data/appdir 
2026-10-11 11:14:28.634 | INFO     | autogenstudio.web.auth.manager:__init__:26 - Initialized auth manager with provider: none
INFO:     Started server process [3588]
INFO:     Waiting for application startup.
2026-10-11 11:14:28.747 | INFO     | autogenstudio.database.db_manager:initialize_database:82 - Creating database tables...
2026-10-11 11:14:28.827 | INFO     | autogenstudio.database.schem

In [6]:
#@title 6. Use AutoGen from Python (optional) — two agents and a tool on the local model
#@markdown Installs `autogen-agentchat` into the notebook's own Python the first time, then runs a small team against the model from cell 3. Edit the code freely.
TASK = "Write a product description for a reusable water bottle in at most 40 words." #@param {type:"string"}
import importlib, importlib.util, sys
if importlib.util.find_spec("autogen_agentchat") is None:
    print("▶ Installing autogen-agentchat (one time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "autogen-agentchat", "autogen-ext[openai]"], check=True)
    importlib.invalidate_caches()

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

MODEL = json.load(open(f"{DATA}/model.json"))
model_client = OpenAIChatCompletionClient(
    model=MODEL["model"], base_url=MODEL["base_url"], api_key=MODEL["api_key"],
    model_info={"vision": False, "function_calling": True, "json_output": True,
                "structured_output": True, "family": "unknown"},
)

async def word_count(text: str) -> int:
    """Count the words in a piece of text."""
    return len(text.split())

writer = AssistantAgent("writer", model_client=model_client, tools=[word_count], reflect_on_tool_use=True,
    system_message="You write short, clear text. When a word limit is given, check your draft with the word_count tool.")
reviewer = AssistantAgent("reviewer", model_client=model_client,
    system_message="You review the writer's latest draft and give short, concrete feedback. "
                   "When the draft is good enough, reply with exactly: APPROVE")
team = RoundRobinGroupChat([writer, reviewer],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(6))

await Console(team.run_stream(task=TASK))
await model_client.close()

▶ Installing autogen-agentchat (one time)...
---------- TextMessage (user) ----------
Write a product description for a reusable water bottle in at most 40 words.
---------- ToolCallRequestEvent (writer) ----------
[FunctionCall(id='call_bzdpdm4h', arguments='{"text":"Eco-friendly and durable, our reusable water bottle keeps your drink cold for hours. Perfect for gym, commute, or everyday use."}', name='word_count'), FunctionCall(id='call_xhizotad', arguments='{"text":"Eco-friendly and durable, our reusable water bottle keeps your drink cold for 12 hours. Perfect for gym, commute, or everyday use."}', name='word_count')]
---------- ToolCallExecutionEvent (writer) ----------
[FunctionExecutionResult(content='20', name='word_count', call_id='call_bzdpdm4h', is_error=False), FunctionExecutionResult(content='21', name='word_count', call_id='call_xhizotad', is_error=False)]
---------- TextMessage (writer) ----------
Eco-friendly and durable, our reusable water bottle keeps your drink cold f

In [ ]:
#@title 7. Keep the runtime alive
import time
while True:
    time.sleep(30)

In [ ]:
#@title 8. Backup to Google Drive
#@markdown Saves the AutoGen Studio database, uploaded files, team configs and the access key to `MyDrive/autogen-backup/`. Run it when no agent run is in progress.
from google.colab import drive
drive.mount("/content/drive")
dst = "/content/drive/MyDrive/autogen-backup"
os.makedirs(dst, exist_ok=True)
subprocess.run(f"tar -czf {dst}/autogen-data.tgz --exclude=./logs --exclude=./model.json --exclude=./seed_teams.py -C {DATA} .",
               shell=True, check=True)
print("Backup written to", f"{dst}/autogen-data.tgz")

In [ ]:
#@title 9. Restore from Google Drive (new runtime: run cells 1–2, then this, then 3–4)
from google.colab import drive
drive.mount("/content/drive")
src = "/content/drive/MyDrive/autogen-backup/autogen-data.tgz"
assert os.path.exists(src), "No backup found"

subprocess.run(["pkill", "-9", "-f", "autogenstudio ui"])
shutil.rmtree(APPDIR, ignore_errors=True)
subprocess.run(f"tar -xzf {src} -C {DATA}", shell=True, check=True)
os.makedirs(f"{APPDIR}/configs", exist_ok=True)
KEYS = json.load(open(f"{DATA}/keys.json"))
print("Restored. Now run cell 3 (model) and cell 4 (Start).")